# SQL Practice - Day 11

## Real Data Analyst Interview Practice

10 mixed questions, slightly harder than Day 10.

Topics: CASE WHEN, Window Functions, LAG, ROW_NUMBER, DENSE_RANK, Self Join, Joins, Correlated Subqueries, CTE, Aggregation on Aggregation.

## Q1

**Task:** Find each customer's total spending and classify them as High (>=5000), Medium (2000-4999), or Low (<2000).

In [1]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,103,104,104],'amount':[1500,2500,3000,700,1200,1000]})
df

,customer_id,amount
0,101,1500
1,101,2500
2,102,3000
3,103,700
4,104,1200
5,104,1000


In [2]:
import pandasql
from pandasql import sqldf
sqldf("""
    With my_cte as (
    select customer_id , sum(amount) as total_spending
    from df
    group by customer_Id
    )
    select customer_id , total_spending,
    case
        when total_spending >= 5000 then 'High'
        when total_spending >= 2000 and total_spending < 4999 then 'Medium'
        else 'Low'
    end as Category
    from my_cte
    
""")

,customer_id,total_spending,Category
0,101,4000,Medium
1,102,3000,Medium
2,103,700,Low
3,104,2200,Medium


## Q2

**Task:** Find the highest-paid employee in each department. If two employees have the same highest salary, return both.

In [3]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5,6],'name':['A','B','C','D','E','F'],'department':['HR','HR','IT','IT','Sales','Sales'],'salary':[60000,70000,90000,90000,80000,70000]})
df

,emp_id,name,department,salary
0,1,A,HR,60000
1,2,B,HR,70000
2,3,C,IT,90000
3,4,D,IT,90000
4,5,E,Sales,80000
5,6,F,Sales,70000


In [4]:
sqldf("""
select * 
        from (select name , department , salary,
        Dense_rank()
        over(partition by department order by salary desc) as rn
        from df) t
where rn = 1        
        



""")

,name,department,salary,rn
0,B,HR,70000,1
1,C,IT,90000,1
2,D,IT,90000,1
3,E,Sales,80000,1


## Q3

**Task:** For each customer, show the current amount and previous order amount using LAG(). Add CASE: Increased, Decreased, or First Order.

In [44]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,101,102,102],'order_date':['2025-01-01','2025-02-01','2025-03-01','2025-01-10','2025-02-10'],'amount':[500,700,600,1000,1200]})
df

,customer_id,order_date,amount
0,101,2025-01-01,500
1,101,2025-02-01,700
2,101,2025-03-01,600
3,102,2025-01-10,1000
4,102,2025-02-10,1200


In [45]:
sqldf("""
SELECT customer_id,
       order_date,
       amount AS current_amount,
       LAG(amount) OVER (
           PARTITION BY customer_id
           ORDER BY order_date
       ) AS previous_amount,
       CASE
           WHEN LAG(amount) OVER (
               PARTITION BY customer_id
               ORDER BY order_date
           ) IS NULL THEN 'First Order'
           WHEN amount > LAG(amount) OVER (
               PARTITION BY customer_id
               ORDER BY order_date
           ) THEN 'Increased'
           WHEN amount < LAG(amount) OVER (
               PARTITION BY customer_id
               ORDER BY order_date
           ) THEN 'Decreased'
           ELSE 'No Change'
       END AS order_status
FROM df
""")

,customer_id,order_date,current_amount,previous_amount,order_status
0,101,2025-01-01,500,NaN,First Order
1,101,2025-02-01,700,500.0,Increased
2,101,2025-03-01,600,700.0,Decreased
3,102,2025-01-10,1000,NaN,First Order
4,102,2025-02-10,1200,1000.0,Increased


## Q4

**Task:** Find customers who placed an order in January 2025 but did NOT place another order in February 2025.

In [7]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5,6],'customer_id':[101,101,102,103,103,104],'order_date':['2025-01-05','2025-02-10','2025-01-15','2025-01-20','2025-03-01','2025-01-25']})
df

,order_id,customer_id,order_date
0,1,101,2025-01-05
1,2,101,2025-02-10
2,3,102,2025-01-15
3,4,103,2025-01-20
4,5,103,2025-03-01
5,6,104,2025-01-25


In [8]:
sqldf("""
SELECT DISTINCT customer_id
FROM df
WHERE customer_id IN (
    SELECT customer_id
    FROM df
    WHERE order_date BETWEEN '2025-01-01' AND '2025-01-31'
)
AND customer_id NOT IN (
    SELECT customer_id
    FROM df
    WHERE order_date BETWEEN '2025-02-01' AND '2025-02-28'
)
""")

,customer_id
0,102
1,103
2,104


## Q5

**Task:** Find products whose price is higher than their category average but lower than the maximum price of their category.

In [9]:
import pandas as pd
df=pd.DataFrame({'product_id':[1,2,3,4,5,6],'category':['A','A','A','B','B','B'],'price':[100,200,300,250,400,500]})
df

,product_id,category,price
0,1,A,100
1,2,A,200
2,3,A,300
3,4,B,250
4,5,B,400
5,6,B,500


In [10]:
sqldf("""
select product_id , category price
from df a 
where price > (select avg(price) 
                from df b
                where a.category = b.category)
and price < (select max(price )
                from df c
                where a.category = c.category)


""")


,product_id,price
0,5,B


## Q6

**Task:** Find employees whose salary is greater than their manager's salary using a SELF JOIN. Return employee and manager details.

In [11]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5],'name':['ManagerA','EmpB','EmpC','ManagerD','EmpE'],'manager_id':[None,1,1,None,4],'salary':[80000,90000,70000,70000,75000]})
df

,emp_id,name,manager_id,salary
0,1,ManagerA,NaN,80000
1,2,EmpB,1.0,90000
2,3,EmpC,1.0,70000
3,4,ManagerD,NaN,70000
4,5,EmpE,4.0,75000


In [12]:
sqldf("""
SELECT
    e.name AS emp_name,
    e.salary AS emp_salary,
    m.name AS manager_name,
    m.salary AS manager_salary
FROM df e
JOIN df m
    ON e.manager_id = m.emp_id
WHERE e.salary > m.salary
""")

,emp_name,emp_salary,manager_name,manager_salary
0,EmpB,90000,ManagerA,80000
1,EmpE,75000,ManagerD,70000


## Q7

**Task:** Find the top 3 customers by total spending. Customers with the same spending must receive the same rank.

In [13]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,103,104,104,105],'amount':[500,500,1200,1500,700,800,1500]})
df

,customer_id,amount
0,101,500
1,101,500
2,102,1200
3,103,1500
4,104,700
5,104,800
6,105,1500


In [14]:
sqldf("""
WITH customer_spending AS (
    SELECT
        customer_id,
        SUM(amount) AS total_spending
    FROM df
    GROUP BY customer_id
),
ranked_customers AS (
    SELECT
        customer_id,
        total_spending,
        DENSE_RANK() OVER (
            ORDER BY total_spending DESC
        ) AS rn
    FROM customer_spending
)
SELECT
    customer_id,
    total_spending,
    rn
FROM ranked_customers
WHERE rn <= 3
ORDER BY rn, customer_id
""")

,customer_id,total_spending,rn
0,103,1500,1
1,104,1500,1
2,105,1500,1
3,102,1200,2
4,101,1000,3


## Q8

**Task:** Find the latest order for every customer using ROW_NUMBER(). Return order_id, customer_id, order_date and amount.

In [15]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5,6],'customer_id':[101,101,102,102,103,103],'order_date':['2025-01-01','2025-03-01','2025-02-01','2025-04-01','2025-01-10','2025-02-10'],'amount':[100,300,500,200,700,800]})
df

,order_id,customer_id,order_date,amount
0,1,101,2025-01-01,100
1,2,101,2025-03-01,300
2,3,102,2025-02-01,500
3,4,102,2025-04-01,200
4,5,103,2025-01-10,700
5,6,103,2025-02-10,800


In [31]:
sqldf("""
select order_id , customer_id , order_date , amount,rn
from (select order_id , customer_id , order_date , amount,
        row_number()
        over(partition by customer_id  order by order_date desc) as rn
        from df)t
where rn = 1
        


""")

,order_id,customer_id,order_date,amount,rn
0,2,101,2025-03-01,300,1
1,4,102,2025-04-01,200,1
2,6,103,2025-02-10,800,1


## Q9

**Task:** Find departments whose maximum salary is greater than the overall average company salary.

In [32]:
import pandas as pd
df=pd.DataFrame({'department':['HR','HR','IT','IT','Sales','Sales'],'salary':[50000,60000,70000,80000,90000,85000]})
df

,department,salary
0,HR,50000
1,HR,60000
2,IT,70000
3,IT,80000
4,Sales,90000
5,Sales,85000


In [35]:
sqldf("""
select department , max(salary) as max_salary
from df
group by department
having max(salary) > (select avg(salary) 
                        from df )


""")

,department,max_salary
0,IT,80000
1,Sales,90000


## Q10

**Task:** Find the second-highest customer by total spending. Return customer_id and total_spending using a CTE and DENSE_RANK().

In [36]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,103,104,104,105],'amount':[500,500,1200,1500,700,800,1500]})
df

,customer_id,amount
0,101,500
1,101,500
2,102,1200
3,103,1500
4,104,700
5,104,800
6,105,1500


In [43]:
sqldf("""
with my_cte as (
select customer_id ,sum(amount) as total_spending,
        dense_rank()
        over(order by sum(amount) desc) as rn
from df 
group by customer_id 

)
select * from my_cte
where rn = 2

""")

,customer_id,total_spending,rn
0,102,1200,2
